# Training_File_Sweep.ipynb — STEP 2B parametric sweep

**Scopo**: orchestrare N esperimenti consecutivi con configurazioni diverse di capacity
(`cf_hidden_size`, `cf_rank`) e scenario (`scenario_mix`, `cut_in_ratio`).

**Differenze rispetto a `Training_File.ipynb`**:
- `Training_File.ipynb` → 1 esperimento singolo, analisi visiva approfondita
- `Training_File_Sweep.ipynb` → N esperimenti in sequenza, summary aggregato

**Strategie chiave**:
1. **Cache condivisa per scenario**: runs con stesso `(n_train, scenario_mix, cut_in_ratio)` riusano la stessa cache → 1 generazione vs N. Capacity sweep su highway-only riusa la cache 5 volte.
2. **Push per-run** (non a fine sweep): se Azure crasha a metà, i results parziali sono già su GitHub.
3. **Continue-on-failure**: se un run fallisce (preflight o training), si logga il problema e si passa al successivo.
4. **Sanity check pre-sweep**: verifica che lo stesso codice non sia già stato runnato (TAG collision check).

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap (deps + nbstripout + sync remote + sanity check)
# ===========================================================
import sys, os

# ── Dep 1: matplotlib (per Cella 5 grafici comparativi) ──
print('📦 Verifica/install matplotlib (richiesto da Cella 5)')
!{sys.executable} -m pip install --quiet matplotlib

# ── Dep 2: nbstripout (evita "local changes would be overwritten") ──
# Configura git per strippare automaticamente gli output dei notebook prima del
# diff. Senza, OGNI esecuzione del notebook crea modifiche locali che bloccano
# `git pull`. Operazione idempotente: --install non duplica se gia' attivo.
print('📦 Verifica/install nbstripout (auto-strip notebook outputs)')
!{sys.executable} -m pip install --quiet nbstripout
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   ✅ nbstripout configurato')

import glob, json, subprocess, datetime, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML, Image

# Sync con remote (essenziale per avere ultimi fix)
print('\n🔄 git pull --no-rebase --no-edit origin main...')
!git pull --no-rebase --no-edit origin main

print('\n📍 Commit attuale:')
!git log --oneline -3

# Sanity check infrastruttura
print('\n🔧 Sanity check:')
!python -c "from train import BatchCSVLogger; from core.network import CF_FSNN_Net; \
            m = CF_FSNN_Net(hidden_size=64, rank=16); \
            assert m.hidden_size == 64 and m.rank == 16, 'sweep API non funzionante'; \
            print('✅ CF_FSNN_Net accetta hidden_size/rank kwargs')"

assert os.path.isfile('scripts/preflight.py'), 'scripts/preflight.py NON trovato'
print('✅ scripts/preflight.py presente')

In [ ]:
# ===========================================================
# CELLA 1 — SWEEP CONFIG: lista esperimenti da eseguire
# ===========================================================
# Ogni elemento è un dict che OVERRIDA i default. Campi minimi richiesti: tag.
# Tutti gli altri usano i default qui sotto.
#
# Strategia STEP 2B:
#   Block A — capacity sweep su highway-only (effetto puro di capacity)
#   Block B — scenario diversity a capacity intermedia (effetto puro di scenario)
#
# Cache condivisione: runs con stesso (n_train, scenario_mix, cut_in_ratio) → stessa
# cache. Block A riusa la stessa cache 5 volte (highway, cut0).

# ── DEFAULTS condivisi (overridabili da ogni run) ──────────────
DEFAULTS = {
    # Training duration — fast iteration mode (STEP 2A validato: convergenza in ~17 min)
    'epochs':       10,
    'n_train':      500,
    'n_val':        100,
    # Scheduler
    'scheduler':    'onecycle',
    'max_lr':       2e-3,
    'T0':           5,
    'lr':           1e-3,
    # Sequenza
    'seq_len':      50,
    'batch_size':   64,
    # Pesi loss PINN
    'lambda_data':  1.0,
    'lambda_phys':  0.1,
    'lambda_ou':    0.05,
    'lambda_bc':    1.0,
    'optimizer':    'adam',
    # Capacity (default = config.py: hidden=32, rank=8)
    'cf_hidden_size': None,   # None → usa CF_HIDDEN_SIZE di config.py (= 32)
    'cf_rank':        None,   # None → usa CF_RANK di config.py        (= 8)
    # Scenario
    'scenario_mix':   'highway',
    'cut_in_ratio':   0.0,
    # Early stopping (aggressivo per fast iteration)
    'early_stop_patience': 2,
    'early_stop_delta':    0.005,
    # Diagnostica
    'max_inf_streak': 20,
}

# ── SWEEP_PLAN: lista degli esperimenti ───────────────────────
SWEEP_PLAN = [
    # ── BLOCK A: capacity sweep, highway-only (cache condivisa) ──
    {'tag': 'P9_S2B_h32_r8_hw',   'cf_hidden_size':  32, 'cf_rank':  8},  # baseline = P9_S2A
    {'tag': 'P9_S2B_h48_r12_hw',  'cf_hidden_size':  48, 'cf_rank': 12},
    {'tag': 'P9_S2B_h64_r16_hw',  'cf_hidden_size':  64, 'cf_rank': 16},
    {'tag': 'P9_S2B_h96_r24_hw',  'cf_hidden_size':  96, 'cf_rank': 24},
    {'tag': 'P9_S2B_h128_r32_hw', 'cf_hidden_size': 128, 'cf_rank': 32},

    # ── BLOCK B: scenario diversity a h64_r16 (presumibile sweet spot) ──
    # Nota: se Block A indicasse un sweet spot diverso, modificare h/r qui.
    {'tag': 'P9_S2B_h64_r16_urban',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'urban'},
    {'tag': 'P9_S2B_h64_r16_truck',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'truck'},
    {'tag': 'P9_S2B_h64_r16_mixed',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'mixed'},  # default 4-scenario
    {'tag': 'P9_S2B_h64_r16_hwcut15', 'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'highway', 'cut_in_ratio': 0.15},
]

# Controllo per-run
RUN_PREFLIGHT  = True    # doppio smoke prima del FULL per ogni run
PUSH_PER_RUN   = True    # push dei results di OGNI run (non solo a fine sweep)
SKIP_IF_EXISTS = True    # se results/<tag>/ esiste già, salta
STOP_ON_FAIL   = False   # False = continua, True = ferma sweep al primo fail

# Report sweep plan
print(f'📋 SWEEP PLAN: {len(SWEEP_PLAN)} esperimenti\n')
for i, run in enumerate(SWEEP_PLAN, 1):
    cfg = {**DEFAULTS, **run}
    print(f"  {i:2d}. {cfg['tag']:<32}  h={cfg['cf_hidden_size']}, r={cfg['cf_rank']}, "
          f"scen={cfg['scenario_mix']}, cut={cfg['cut_in_ratio']}")

print(f'\n⚙️  Flags: preflight={RUN_PREFLIGHT}, push_per_run={PUSH_PER_RUN}, '
      f'skip_if_exists={SKIP_IF_EXISTS}, stop_on_fail={STOP_ON_FAIL}')
print(f'⏱️  Tempo stimato Azure CPU: ~5h totali (ciascun run 17-75 min in base a capacity)')

In [ ]:
# ===========================================================
# CELLA 2 — Helper: cache path normalizer + run executor
# ===========================================================
def _safe_scenario(s):
    """Normalizza scenario_mix per usarlo in un nome file.
    'highway:0.7,urban:0.3' → 'highway_0.7_urban_0.3'
    """
    return s.replace(':', '_').replace(',', '_').replace(' ', '')

def _cache_path(cfg):
    """Path cache deterministico da (n_train, scenario, cut_in). Runs con stessa
    config dataset riusano la stessa cache anche con h/r diversi."""
    s = _safe_scenario(cfg['scenario_mix'])
    return f"data/cache_{cfg['n_train']}_{s}_cut{cfg['cut_in_ratio']}.pt"

def _build_cli_args(cfg, cache_path):
    """Costruisce gli argv di train.py da un cfg dict."""
    args = [
        '--epochs',         str(cfg['epochs']),
        '--scheduler',      cfg['scheduler'],
        '--seq_len',        str(cfg['seq_len']),
        '--batch_size',     str(cfg['batch_size']),
        '--lambda_data',    str(cfg['lambda_data']),
        '--lambda_phys',    str(cfg['lambda_phys']),
        '--lambda_ou',      str(cfg['lambda_ou']),
        '--lambda_bc',      str(cfg['lambda_bc']),
        '--optimizer',      cfg['optimizer'],
        '--max_inf_streak', str(cfg['max_inf_streak']),
        '--data_cache',     cache_path,
        '--tag',            cfg['tag'],
        '--n_train',        str(cfg['n_train']),
        '--n_val',          str(cfg['n_val']),
        '--scenario_mix',   cfg['scenario_mix'],
        '--early_stop_patience', str(cfg['early_stop_patience']),
        '--early_stop_delta',    str(cfg['early_stop_delta']),
    ]
    if cfg.get('cut_in_ratio') is not None:
        args += ['--cut_in_ratio', str(cfg['cut_in_ratio'])]
    if cfg.get('cf_hidden_size') is not None:
        args += ['--cf_hidden_size', str(cfg['cf_hidden_size'])]
    if cfg.get('cf_rank') is not None:
        args += ['--cf_rank', str(cfg['cf_rank'])]
    if cfg.get('lambda_sr') is not None:
        args += ['--lambda_sr', str(cfg['lambda_sr'])]
    # Scheduler-specific
    if cfg['scheduler'] == 'onecycle':
        args += ['--max_lr', str(cfg['max_lr'])]
    elif cfg['scheduler'] == 'cosine':
        args += ['--T0', str(cfg['T0'])]
    elif cfg['scheduler'] == 'plateau':
        args += ['--lr', str(cfg['lr'])]
    return args

def _push_results(cfg):
    """Copia checkpoints/<tag>/ → results/<tag>/, commit+pull-before-push.
    
    Funziona anche su training FAILED (artefatti dal batch_log persistono).
    Commit message allineato a Training_File.ipynb Cella 8 per coerenza
    (status, batch totali, first_inf, gn_max, spike_rate).
    
    Robusto al fatto che il kernel Jupyter su Azure NON ha torch installato:
    nessun import torch qui. CRASH_INFO viene scritto solo dai dati presenti
    nei CSV (training_log e batch_log), non dal crash_model.pt.
    """
    import numpy as np, pandas as pd
    
    tag     = cfg['tag']
    src_dir = f'checkpoints/{tag}'
    dst_dir = f'results/{tag}'
    if not os.path.isdir(src_dir):
        print(f'   ⚠️  {src_dir} mancante — niente da pushare')
        return False

    # Copia CSV + JSON + PNG (NO .pt: pesi restano in checkpoints/)
    if os.path.isdir(dst_dir):
        shutil.rmtree(dst_dir)
    os.makedirs(f'{dst_dir}/plots', exist_ok=True)
    for f in glob.glob(f'{src_dir}/*.csv') + glob.glob(f'{src_dir}/*.json'):
        shutil.copy2(f, dst_dir)
    for f in glob.glob(f'{src_dir}/plots/*.png'):
        shutil.copy2(f, f'{dst_dir}/plots/')
    
    # Crash info SENZA torch: rilevata da presenza di crash_model.pt + dati dai CSV.
    # In passato leggevamo epoch/val_loss da crash_model.pt con torch.load, ma il
    # kernel Jupyter Azure (azureml_py38 oppure altro) potrebbe non avere torch.
    # Sono dati che ricaviamo comunque dal training_log.csv.
    crash_pt = f'{src_dir}/crash_model.pt'
    if os.path.isfile(crash_pt):
        epoch_csv = f'{src_dir}/training_log.csv'
        crash_info = ['Crash detected (vedi training_log.csv per dettagli)']
        if os.path.isfile(epoch_csv):
            edf = pd.read_csv(epoch_csv)
            if len(edf) > 0:
                best_idx = int(edf['val_total'].idxmin())
                crash_info.append(f'Last epoch run: {len(edf)}')
                crash_info.append(f'Best val_loss:  {edf["val_total"].min():.5f} (E{best_idx+1})')
        with open(f'{dst_dir}/CRASH_INFO.txt', 'w') as f:
            f.write('\n'.join(crash_info) + '\n')
    
    # Analisi batch_log per commit message (come Training_File.ipynb Cella 8)
    batch_csv = f'{dst_dir}/training_batch_log.csv'
    msg_extra = ''
    if os.path.isfile(batch_csv):
        bdf = pd.read_csv(batch_csv)
        gn_pre = bdf['gn_total_preclip'].replace([np.inf, -np.inf], np.nan)
        first_inf_list = bdf.index[bdf['is_inf_grad'] == 1].tolist()
        first_inf = int(first_inf_list[0]) + 1 if first_inf_list else None
        status = 'FAILED' if first_inf else 'OK'
        msg_extra = (
            f"\nStatus:         {status}\n"
            f"Batch totali:   {len(bdf)}\n"
            f"Primo inf grad: {f'B{first_inf}' if first_inf else 'nessuno'}\n"
            f"Inf totali:     {int(bdf['is_inf_grad'].sum())}\n"
            f"gn max finito:  {gn_pre.max():.3e}\n"
            f"Spike rate:     {bdf['spike_rate'].mean()*100:.2f}%\n"
        )
    
    # Best val_loss da training_log.csv (se presente)
    epoch_csv = f'{dst_dir}/training_log.csv'
    val_str = ''
    if os.path.isfile(epoch_csv):
        edf = pd.read_csv(epoch_csv)
        if len(edf) > 0:
            val_str = f"Best val_loss:  {edf['val_total'].min():.4f} (E{int(edf['val_total'].idxmin())+1}/{len(edf)})\n"
    
    ts  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f"results: {tag} ({ts})\n\n"
        f"Sweep run via Training_File_Sweep.ipynb\n"
        f"Capacity: hidden={cfg.get('cf_hidden_size')}, rank={cfg.get('cf_rank')}\n"
        f"Scenario: {cfg['scenario_mix']}, cut_in={cfg['cut_in_ratio']}\n"
        f"Config:   scheduler={cfg['scheduler']}, max_lr={cfg.get('max_lr','-')}, "
        f"seq_len={cfg['seq_len']}, epochs={cfg['epochs']}, n_train={cfg['n_train']}\n"
        f"{val_str}{msg_extra}\n"
        f"Artefatti: CSV (per-epoca + per-batch) + plots G1-G13.\n"
        f"Pesi modello restano in checkpoints/{tag}/ su Azure.\n"
    )
    with open('/tmp/sweep_commit_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    
    !git add {dst_dir}
    !git commit -F /tmp/sweep_commit_msg.txt
    !rm /tmp/sweep_commit_msg.txt
    print('   🔄 git pull --no-rebase --no-edit origin main')
    !git pull --no-rebase --no-edit origin main
    print('   📤 git push origin main')
    !git push origin main
    return True

print('✅ Helper functions definite')

In [ ]:
# ===========================================================
# CELLA 3 — ESECUZIONE SWEEP
# Per ogni run: preflight (opzionale) → train.py → push results (opzionale)
# Robust: continue-on-failure (default), tracking di sweep state in DataFrame.
# ===========================================================
sweep_results = []   # raccoglie risultati per il summary finale
sweep_start  = time.time()

for i, run_override in enumerate(SWEEP_PLAN, 1):
    cfg = {**DEFAULTS, **run_override}
    tag = cfg['tag']
    
    print('\n' + '=' * 78)
    print(f'🚀 RUN {i}/{len(SWEEP_PLAN)}: {tag}')
    print(f'   Capacity: hidden={cfg["cf_hidden_size"]}, rank={cfg["cf_rank"]}')
    print(f'   Scenario: {cfg["scenario_mix"]}, cut_in={cfg["cut_in_ratio"]}')
    print(f'   Cache:    {_cache_path(cfg)}')
    print('=' * 78)
    
    run_start = time.time()
    record = {
        'tag': tag,
        'cf_hidden_size': cfg['cf_hidden_size'],
        'cf_rank':        cfg['cf_rank'],
        'scenario_mix':   cfg['scenario_mix'],
        'cut_in_ratio':   cfg['cut_in_ratio'],
        'status':         'unknown',
        'elapsed_s':      0.0,
        'best_val':       np.nan,
        'best_epoch':     np.nan,
        'n_epochs':       0,
        'n_inf_batches':  0,
        'spike_pct':      np.nan,
    }
    
    # ── Skip se results/<tag>/ esiste già (riprese di sweep parziali) ────
    if SKIP_IF_EXISTS and os.path.isdir(f'results/{tag}'):
        print(f'   ⏭️  SKIP: results/{tag}/ già presente (set SKIP_IF_EXISTS=False per forzare)')
        record['status'] = 'skipped_existing'
        sweep_results.append(record)
        continue
    
    cache_path = _cache_path(cfg)
    cli_args   = _build_cli_args(cfg, cache_path)
    
    # ── PREFLIGHT (doppio smoke) ─────────────────────────────────────
    # FIX STEP 2B: passare scenario_mix + cut_in_ratio + h/r al preflight,
    # altrimenti gli smoke girano su default invece dello scenario pianificato
    # (bug scoperto al primo lancio sweep — vedi P_S.md).
    preflight_ok = True
    if RUN_PREFLIGHT:
        pf_extra = ['--seq_len', str(cfg['seq_len'])]
        if cfg['scheduler'] == 'onecycle':
            pf_extra += ['--max_lr', str(cfg['max_lr'])]
        if cfg['cf_hidden_size'] is not None:
            pf_extra += ['--cf_hidden_size', str(cfg['cf_hidden_size'])]
        if cfg['cf_rank'] is not None:
            pf_extra += ['--cf_rank', str(cfg['cf_rank'])]
        # Scenario propagato anche allo smoke
        pf_extra += ['--scenario_mix', cfg['scenario_mix']]
        if cfg.get('cut_in_ratio') is not None:
            pf_extra += ['--cut_in_ratio', str(cfg['cut_in_ratio'])]
        pf_cmd = ['python', 'scripts/preflight.py', '--base_tag', tag, '--extra'] + pf_extra
        print(f'\n🚦 Preflight:  {" ".join(pf_cmd)}')
        pf_res = subprocess.run(pf_cmd, capture_output=False)
        preflight_ok = (pf_res.returncode == 0)
        if not preflight_ok:
            print(f'   ❌ PREFLIGHT FAIL — skipping FULL train per questo run')
            record['status']    = 'preflight_fail'
            record['elapsed_s'] = time.time() - run_start
            sweep_results.append(record)
            if STOP_ON_FAIL:
                print('   🛑 STOP_ON_FAIL=True → sweep interrotto')
                break
            continue
    
    # ── FULL TRAIN ──────────────────────────────────────────────────
    full_cmd = ['python', 'train.py'] + cli_args
    print(f'\n▶️  Full train: {" ".join(full_cmd)}')
    full_res = subprocess.run(full_cmd, capture_output=False)
    full_ok  = (full_res.returncode == 0)
    
    # Estrai metriche anche se FULL FAIL (potrebbe avere salvato qualche epoca)
    epoch_csv = f'checkpoints/{tag}/training_log.csv'
    batch_csv = f'checkpoints/{tag}/training_batch_log.csv'
    if os.path.isfile(epoch_csv):
        edf = pd.read_csv(epoch_csv)
        if len(edf) > 0:
            record['best_val']   = float(edf['val_total'].min())
            record['best_epoch'] = int(edf['val_total'].idxmin()) + 1
            record['n_epochs']   = int(len(edf))
    if os.path.isfile(batch_csv):
        bdf = pd.read_csv(batch_csv)
        record['n_inf_batches'] = int(bdf['is_inf_grad'].sum())
        record['spike_pct']     = float(bdf['spike_rate'].mean() * 100)
    
    record['status']    = 'ok' if full_ok else 'full_fail'
    record['elapsed_s'] = time.time() - run_start
    
    # ── PUSH PER-RUN ────────────────────────────────────────────────
    if PUSH_PER_RUN:
        print(f'\n📤 Push results/{tag}/...')
        _push_results(cfg)
    
    # Formatta best_val pre-print per evitare f-string conditional invalido
    best_val_str = f'{record["best_val"]:.4f}' if not np.isnan(record['best_val']) else 'n/a'
    print(f'\n✅ Run {i}/{len(SWEEP_PLAN)} completato — status={record["status"]}, '
          f'best_val={best_val_str}, elapsed={record["elapsed_s"]/60:.1f}min')
    sweep_results.append(record)
    
    if not full_ok and STOP_ON_FAIL:
        print('   🛑 STOP_ON_FAIL=True → sweep interrotto')
        break

sweep_total_min = (time.time() - sweep_start) / 60
print('\n' + '=' * 78)
print(f'🏁 SWEEP COMPLETATO — {len(sweep_results)}/{len(SWEEP_PLAN)} runs in {sweep_total_min:.1f} min')
print('=' * 78)

In [ ]:
# ===========================================================
# CELLA 4 — SUMMARY: tabella riassuntiva sweep
# ===========================================================
if not sweep_results:
    print('❌ Nessun risultato — esegui Cella 3 prima.')
else:
    df = pd.DataFrame(sweep_results)
    
    # Formatting
    df_disp = df.copy()
    df_disp['elapsed_min'] = (df_disp['elapsed_s'] / 60).round(1)
    df_disp['best_val']    = df_disp['best_val'].apply(lambda x: f'{x:.4f}' if pd.notna(x) else '-')
    df_disp['spike_pct']   = df_disp['spike_pct'].apply(lambda x: f'{x:.2f}%' if pd.notna(x) else '-')
    df_disp['epochs']      = df_disp['n_epochs'].astype(str) + '/' + df_disp.apply(
        lambda r: '?', axis=1)
    
    cols = ['tag','status','cf_hidden_size','cf_rank','scenario_mix','cut_in_ratio',
            'best_val','best_epoch','n_epochs','n_inf_batches','spike_pct','elapsed_min']
    display(Markdown(f'### 📊 SWEEP SUMMARY (N={len(df)})'))
    display(df_disp[cols])
    
    # ── Best per category ────────────────────────────────────────
    ok_runs = df[df['status'] == 'ok']
    if len(ok_runs) > 0:
        display(Markdown('### 🏆 Highlights'))
        
        # Capacity sweep (highway-only)
        cap_runs = ok_runs[(ok_runs['scenario_mix'] == 'highway') & (ok_runs['cut_in_ratio'] == 0.0)]
        if len(cap_runs) > 0:
            best_cap = cap_runs.loc[cap_runs['best_val'].idxmin()]
            print(f"   🎯 Best capacity (highway):  {best_cap['tag']:<35} "
                  f"h={best_cap['cf_hidden_size']}, r={best_cap['cf_rank']}, "
                  f"val={best_cap['best_val']:.4f}")
            print(f"\n   📈 Capacity sweep (highway):")
            for _, r in cap_runs.sort_values('cf_hidden_size').iterrows():
                bar = '█' * int((0.30 - min(r['best_val'], 0.30)) * 200)
                print(f"     h={r['cf_hidden_size']:>3}, r={r['cf_rank']:>2}: "
                      f"val={r['best_val']:.4f} {bar}")
        
        # Scenario diversity
        scn_runs = ok_runs[~((ok_runs['scenario_mix'] == 'highway') & (ok_runs['cut_in_ratio'] == 0.0))]
        if len(scn_runs) > 0:
            print(f"\n   🌍 Scenario diversity:")
            for _, r in scn_runs.iterrows():
                print(f"     {r['scenario_mix']:<10} cut={r['cut_in_ratio']:.2f}: val={r['best_val']:.4f}")
    
    # ── Fail analysis ────────────────────────────────────────────
    fails = df[~df['status'].isin(['ok','skipped_existing'])]
    if len(fails) > 0:
        display(Markdown(f'### ⚠️  Fails: {len(fails)}'))
        display(fails[['tag','status','n_epochs','n_inf_batches','elapsed_min' if 'elapsed_min' in fails.columns else 'elapsed_s']])
    
    # Salva CSV per analisi offline
    sweep_csv = f'sweep_summary_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}.csv'
    df.to_csv(sweep_csv, index=False)
    print(f'\n💾 Summary salvato in: {sweep_csv}')

In [ ]:
# ===========================================================
# CELLA 5 — GRAFICI di confronto sweep
# Legge results/<tag>/ per ogni tag dello SWEEP_PLAN.
# Robusto a sweep parziali: lavora su quanto c'e' in results/.
# Salva i grafici in sweep_plots/<timestamp>/ e li mostra inline.
# ===========================================================
import os, json, glob, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Output dir per i plot di sweep
sweep_plot_dir = f"sweep_plots/{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(sweep_plot_dir, exist_ok=True)

# ── Carica tutti i tag presenti nello SWEEP_PLAN che hanno results/ ──
sweep_data = []   # lista di dict con epoch_df + batch_df + cfg per ogni run OK
for run_override in SWEEP_PLAN:
    cfg = {**DEFAULTS, **run_override}
    tag = cfg['tag']
    res_dir = f'results/{tag}'
    if not os.path.isdir(res_dir):
        continue
    
    epoch_csv = f'{res_dir}/training_log.csv'
    batch_csv = f'{res_dir}/training_batch_log.csv'
    cfg_json  = f'{res_dir}/config_snapshot.json'
    
    if not os.path.isfile(epoch_csv):
        continue
    
    edf = pd.read_csv(epoch_csv)
    if len(edf) == 0:
        continue
    
    bdf = pd.read_csv(batch_csv) if os.path.isfile(batch_csv) else None
    saved_cfg = json.load(open(cfg_json)) if os.path.isfile(cfg_json) else cfg
    
    sweep_data.append({
        'tag': tag,
        'cfg': cfg,
        'saved_cfg': saved_cfg,
        'epoch_df': edf,
        'batch_df': bdf,
        # Categoria: 'capacity' = Block A (highway-only), 'scenario' = Block B
        'category': 'capacity' if (cfg['scenario_mix'] == 'highway' and cfg['cut_in_ratio'] == 0.0) else 'scenario',
    })

if not sweep_data:
    print('❌ Nessun result presente in results/<tag>/ per i tag dello SWEEP_PLAN.')
    print('   Esegui Cella 3 (sweep) prima.')
else:
    print(f'📊 Trovati {len(sweep_data)} runs con results — generazione grafici comparativi...\n')
    
    # Helper: colore per run (gradiente per capacity, distinto per scenario)
    cap_runs = [d for d in sweep_data if d['category'] == 'capacity']
    scn_runs = [d for d in sweep_data if d['category'] == 'scenario']
    cap_runs_sorted = sorted(cap_runs, key=lambda d: d['cfg']['cf_hidden_size'] or 32)
    cmap_cap = plt.cm.viridis(np.linspace(0.15, 0.9, max(len(cap_runs_sorted), 1)))
    cmap_scn = plt.cm.tab10(np.arange(max(len(scn_runs), 1)))
    
    color_map = {}
    for i, d in enumerate(cap_runs_sorted):
        color_map[d['tag']] = cmap_cap[i]
    for i, d in enumerate(scn_runs):
        color_map[d['tag']] = cmap_scn[i % 10]
    
    # ============================================================
    # S1 — Capacity sweep: val_loss vs hidden_size (Block A)
    # ============================================================
    if len(cap_runs_sorted) >= 2:
        fig, ax = plt.subplots(figsize=(9, 5))
        xs   = [d['cfg']['cf_hidden_size'] or 32 for d in cap_runs_sorted]
        vals = [d['epoch_df']['val_total'].min() for d in cap_runs_sorted]
        n_pars = []
        for d in cap_runs_sorted:
            h = d['cfg']['cf_hidden_size'] or 32
            r = d['cfg']['cf_rank']        or 8
            # CF_FSNN_Net params = 4*h + h*r + r*h + h + h*5 + 5 (approssimativo)
            n_pars.append(4*h + 2*h*r + 2*h + h*5 + 5)
        ax.plot(xs, vals, 'o-', linewidth=2, markersize=10, color='#1f77b4', label='best val_loss')
        for x, v, n in zip(xs, vals, n_pars):
            ax.annotate(f'{v:.4f}\n({n:,}p)', (x, v), textcoords='offset points',
                        xytext=(0, 10), ha='center', fontsize=9)
        ax.set_xlabel('hidden_size (rank=hidden/4)', fontsize=11)
        ax.set_ylabel('best val_total (lower=better)', fontsize=11)
        ax.set_title('S1 — Capacity sweep: val_loss vs hidden_size (highway-only)', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(xs)
        # Annotation best
        best_idx = int(np.argmin(vals))
        ax.annotate(f'★ BEST: h={xs[best_idx]}', (xs[best_idx], vals[best_idx]),
                    textcoords='offset points', xytext=(20, -25), ha='left',
                    fontsize=11, fontweight='bold', color='red',
                    arrowprops=dict(arrowstyle='->', color='red'))
        plt.tight_layout()
        out_path = f'{sweep_plot_dir}/S1_capacity_sweep.png'
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'   💾 {out_path}')
    
    # ============================================================
    # S2 — Training curves overlay: val_total per epoca
    # ============================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pannello sinistro: Block A (capacity)
    for d in cap_runs_sorted:
        ep = d['epoch_df']
        h  = d['cfg']['cf_hidden_size'] or 32
        ax1.plot(ep['epoch'], ep['val_total'], 'o-', linewidth=2,
                 color=color_map[d['tag']], label=f"h={h}")
    ax1.set_xlabel('epoch'); ax1.set_ylabel('val_total')
    ax1.set_title('S2a — Capacity sweep training curves (highway)')
    ax1.grid(True, alpha=0.3); ax1.legend(loc='upper right', fontsize=9)
    
    # Pannello destro: Block B (scenario)
    for d in scn_runs:
        ep = d['epoch_df']
        lbl = f"{d['cfg']['scenario_mix']}_cut{d['cfg']['cut_in_ratio']:.2f}"
        ax2.plot(ep['epoch'], ep['val_total'], 's-', linewidth=2,
                 color=color_map[d['tag']], label=lbl)
    # Reference: best capacity highway-only
    if cap_runs_sorted:
        best = min(cap_runs_sorted, key=lambda d: d['epoch_df']['val_total'].min())
        ep = best['epoch_df']
        ax2.plot(ep['epoch'], ep['val_total'], 'k--', linewidth=1.5, alpha=0.6,
                 label=f"REF: {best['tag']}")
    ax2.set_xlabel('epoch'); ax2.set_ylabel('val_total')
    ax2.set_title('S2b — Scenario diversity training curves (h64_r16)')
    ax2.grid(True, alpha=0.3); ax2.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    out_path = f'{sweep_plot_dir}/S2_training_curves.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 {out_path}')
    
    # ============================================================
    # S3 — Bar chart: best val_loss per run
    # ============================================================
    fig, ax = plt.subplots(figsize=(12, 5))
    all_runs = cap_runs_sorted + scn_runs
    labels   = []
    bests    = []
    colors   = []
    for d in all_runs:
        bests.append(d['epoch_df']['val_total'].min())
        if d['category'] == 'capacity':
            labels.append(f"h{d['cfg']['cf_hidden_size']}_hw")
            colors.append('#1f77b4')
        else:
            labels.append(f"{d['cfg']['scenario_mix']}_cut{d['cfg']['cut_in_ratio']:.2f}")
            colors.append('#ff7f0e')
    bars = ax.bar(range(len(labels)), bests, color=colors, edgecolor='black')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('best val_total'); ax.set_title('S3 — Best val_total per esperimento')
    ax.grid(True, alpha=0.3, axis='y')
    # Linea baseline 0.2802 (P9_S2A_fast_baseline)
    ax.axhline(0.2802, ls='--', color='red', alpha=0.6, label='P9_S2A baseline (0.2802)')
    # Valori sopra barre
    for bar, v in zip(bars, bests):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}',
                ha='center', fontsize=8)
    # Legenda categoria
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#1f77b4', label='Block A (capacity, highway)'),
        Patch(facecolor='#ff7f0e', label='Block B (scenario)'),
        plt.Line2D([0],[0], color='red', ls='--', label='P9_S2A baseline'),
    ], loc='upper right', fontsize=9)
    plt.tight_layout()
    out_path = f'{sweep_plot_dir}/S3_bar_best_val.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 {out_path}')
    
    # ============================================================
    # S4 — Spike rate vs capacity (sparsity behavior)
    # ============================================================
    if any(d['batch_df'] is not None for d in cap_runs_sorted):
        fig, ax = plt.subplots(figsize=(9, 5))
        xs    = []
        means = []
        stds  = []
        for d in cap_runs_sorted:
            if d['batch_df'] is None:
                continue
            xs.append(d['cfg']['cf_hidden_size'] or 32)
            means.append(d['batch_df']['spike_rate'].mean() * 100)
            stds.append(d['batch_df']['spike_rate'].std() * 100)
        ax.errorbar(xs, means, yerr=stds, fmt='o-', linewidth=2, markersize=10,
                    capsize=5, color='#2ca02c')
        ax.axhline(15, ls='--', color='red', alpha=0.6, label='target 15%')
        ax.axhspan(10, 25, alpha=0.1, color='green', label='target band 10-25%')
        ax.set_xlabel('hidden_size'); ax.set_ylabel('spike_rate %')
        ax.set_title('S4 — Spike rate vs capacity (mean ± std per-batch)')
        ax.grid(True, alpha=0.3); ax.legend(loc='best', fontsize=9)
        ax.set_xticks(xs)
        plt.tight_layout()
        out_path = f'{sweep_plot_dir}/S4_spike_vs_capacity.png'
        plt.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'   💾 {out_path}')
    
    # ============================================================
    # S5 — Pareto: val_loss vs training time (efficiency)
    # ============================================================
    fig, ax = plt.subplots(figsize=(9, 5))
    for d in cap_runs_sorted:
        ep    = d['epoch_df']
        t_min = ep['time_s'].sum() / 60
        best  = ep['val_total'].min()
        h     = d['cfg']['cf_hidden_size'] or 32
        ax.scatter(t_min, best, s=200, color=color_map[d['tag']],
                   edgecolor='black', label=f'h={h}', zorder=3)
        ax.annotate(f'h={h}', (t_min, best), textcoords='offset points',
                    xytext=(8, 8), fontsize=9)
    for d in scn_runs:
        ep    = d['epoch_df']
        t_min = ep['time_s'].sum() / 60
        best  = ep['val_total'].min()
        lbl   = f"{d['cfg']['scenario_mix']}"
        ax.scatter(t_min, best, s=180, marker='s', color=color_map[d['tag']],
                   edgecolor='black', zorder=3)
        ax.annotate(lbl, (t_min, best), textcoords='offset points',
                    xytext=(8, -15), fontsize=8, color='darkorange')
    ax.set_xlabel('training time (min)')
    ax.set_ylabel('best val_total')
    ax.set_title('S5 — Pareto: val_loss vs training time (◯=capacity, ◻=scenario)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = f'{sweep_plot_dir}/S5_pareto.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 {out_path}')
    
    # ============================================================
    # S6 — Convergence speed heatmap: epoch in cui si raggiunge il best
    # ============================================================
    fig, ax = plt.subplots(figsize=(10, 4))
    tags        = [d['tag'] for d in cap_runs_sorted + scn_runs]
    best_epochs = [int(d['epoch_df']['val_total'].idxmin()) + 1 for d in cap_runs_sorted + scn_runs]
    n_epochs    = [int(len(d['epoch_df']))                       for d in cap_runs_sorted + scn_runs]
    matrix = np.zeros((1, len(tags)))
    for i, (be, ne) in enumerate(zip(best_epochs, n_epochs)):
        matrix[0, i] = be / ne if ne > 0 else 0
    im = ax.imshow(matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(tags)))
    ax.set_xticklabels([t.replace('P9_S2B_','') for t in tags], rotation=45, ha='right')
    ax.set_yticks([0]); ax.set_yticklabels(['best_epoch / n_epochs'])
    for i, (be, ne) in enumerate(zip(best_epochs, n_epochs)):
        ax.text(i, 0, f'E{be}/{ne}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax, label='fraction (0=convergenza immediata, 1=migliora fino alla fine)')
    ax.set_title('S6 — Convergence speed: dove si trova il best epoch')
    plt.tight_layout()
    out_path = f'{sweep_plot_dir}/S6_convergence.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'   💾 {out_path}')
    
    # ── Summary printout ────────────────────────────────────────
    display(Markdown(f'### 📁 Grafici salvati in `{sweep_plot_dir}/`'))
    display(Markdown(
        '**Legenda grafici sweep:**\n'
        '- **S1**: val_loss vs hidden_size — vedi se c\'è plateau capacity (P9 risolto?) o monotono in calo\n'
        '- **S2**: training curves overlay — vedi se runs grandi convergono più lentamente o sovrappongono\n'
        '- **S3**: bar chart con linea baseline 0.2802 — quali esperimenti la battono\n'
        '- **S4**: spike_rate vs capacity — sparsity si stabilizza al crescere della rete?\n'
        '- **S5**: Pareto val_loss vs time — sweet spot accuracy/efficiency\n'
        '- **S6**: heatmap best_epoch/n_epochs — early stopping equilibrato?\n'
    ))
    
    # Auto-suggestion in base ai risultati
    if len(cap_runs_sorted) >= 3:
        cap_vals = [(d['cfg']['cf_hidden_size'] or 32, d['epoch_df']['val_total'].min())
                    for d in cap_runs_sorted]
        cap_vals.sort()
        best_h, best_v = min(cap_vals, key=lambda x: x[1])
        baseline_v = next((v for h, v in cap_vals if h == 32), None)
        if baseline_v and best_v < baseline_v - 0.005:
            display(Markdown(
                f'### 🎯 Auto-diagnosi: **P9 confermato risolvibile da capacity**\n'
                f'- Best capacity: h={best_h} (val={best_v:.4f}) vs baseline h=32 (val={baseline_v:.4f})\n'
                f'- Δ = {baseline_v - best_v:.4f} → STEP 2C: definitivo a h={best_h} + fine-tuning λ\n'
            ))
        elif baseline_v and abs(best_v - baseline_v) < 0.005:
            display(Markdown(
                f'### ⚠️  Auto-diagnosi: **plateau anche post-capacity**\n'
                f'- Best={best_v:.4f}, baseline h=32={baseline_v:.4f}: capacity NON e\' il bottleneck\n'
                f'- Possibili cause: dataset diversity, loss formulation, surrogate γ, o quantization Po2\n'
                f'- Next: indagare loss components (L_phys vs L_data) e scenario diversity (Block B)\n'
            ))

In [ ]:
# ===========================================================
# CELLA 6 — Push finale: sweep_summary CSV + plot S1-S6 su git
# Pusha aggregati del sweep (NON pushati dai per-run di Cella 3).
# Skippa se non c'e' niente da pushare.
# ===========================================================
import os, glob, datetime, subprocess

# Trova l'ultimo sweep_plots/<timestamp>/ generato da Cella 5
plot_dirs = sorted(glob.glob('sweep_plots/*'), key=os.path.getmtime)
latest_plots = plot_dirs[-1] if plot_dirs else None

# Trova l'ultimo sweep_summary_*.csv (Cella 4)
summary_csvs = sorted(glob.glob('sweep_summary_*.csv'), key=os.path.getmtime)
latest_summary = summary_csvs[-1] if summary_csvs else None

if not latest_plots and not latest_summary:
    print('⏭️  Nessun aggregato da pushare (esegui Cella 4 e/o 5 prima)')
else:
    print(f'📦 Aggregati da pushare:')
    files_to_add = []
    if latest_summary:
        print(f'   • {latest_summary}')
        files_to_add.append(latest_summary)
    if latest_plots:
        n_pngs = len(glob.glob(f'{latest_plots}/*.png'))
        print(f'   • {latest_plots}/ ({n_pngs} PNG)')
        files_to_add.append(latest_plots)
    
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f"sweep aggregates ({ts})\n\n"
        f"Aggregati di confronto cross-run da Training_File_Sweep.ipynb:\n"
        + (f"- {latest_summary} (tabella aggregata)\n" if latest_summary else '')
        + (f"- {latest_plots}/ ({n_pngs} PNG plot comparativi S1-S6)\n" if latest_plots else '')
        + "\nI results per-run sono gia' stati pushati dal loop di Cella 3.\n"
    )
    with open('/tmp/sweep_agg_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    
    for path in files_to_add:
        subprocess.run(['git', 'add', path], check=False)
    
    print('\n📝 git commit:')
    !git commit -F /tmp/sweep_agg_msg.txt
    !rm /tmp/sweep_agg_msg.txt
    print('\n🔄 git pull --no-rebase --no-edit origin main:')
    !git pull --no-rebase --no-edit origin main
    print('\n📤 git push origin main:')
    !git push origin main
    print(f'\n✅ Aggregati pushati. Domani mattina, in locale: git pull && ls {latest_plots}/')